In [ ]:
import pandas as pd
import numpy as np
import re
import scipy.stats
import seaborn as sns
import matplotlib.pyplot as plt
import os

In [ ]:
# df5 = pd.read_csv("pubmed_longdocfactscore.csv")
# df5['model_3_factuality']

In [ ]:
df1 = pd.read_csv("./data/DialogSum/CoVe_ldfacts.csv")
df1.head()

In [ ]:
# df1.columns

In [ ]:
df2 = pd.read_json("./data/DialogSum/summary_LDfacts.json")
df2.rename(columns={'summary_ldfacts_doc_hyp': 'human_summary_factuality'}, inplace=True)
merged_df = pd.merge(df1, df2, on='id', how='inner')
merged_df.head()

#### Kendall Tau Matrix

In [ ]:
def get_df(df):
    cols = [col for col in df.columns if any(metric in col for metric in ["factuality", "rouge", "bleu", "meteor", "bertscore", "bartscore", "ldfacts"])]

    metrics = set([re.sub(r'^.*summary_', '', col) for col in cols])
    df_new = pd.DataFrame()

    for metric in metrics:
        cols_for_metric = [col_name for col_name in df.columns if metric in col_name]
        scores = []
        for col_for_metric in cols_for_metric:
            scores += list(df[col_for_metric])
        scores = scores[:len(df)]
        df_new[metric] = scores

    if "Human_summary_factuality" in df.columns:
        df_new["factuality"] = df["Human_summary_factuality"].values

    df_new = df_new.reset_index(drop=True)
    return df_new

In [ ]:
df_new = get_df(merged_df)
print(df_new.columns)

In [ ]:
def calculate_means_and_correlation(df):
    import re
    import pandas as pd
    import seaborn as sns
    import matplotlib.pyplot as plt

    # Filter columns based on specific metrics
    cols = [col for col in df.columns if any(metric in col for metric in ["factuality", "rouge", "bleu", "meteor", "bertscore", "ldfacts", "bartscore"])]
    
    # Initialize a DataFrame to store the mean values
    df_means = pd.DataFrame()

    if "human_summary_factuality" in df.columns:
        df_means["factuality"] = df["human_summary_factuality"].values

    for col in cols:
        metric_name = re.sub(r'^.*summary_', '', col)
        if metric_name not in df_means:
            df_means[metric_name] = df[[col]].mean(axis=1)
        else:
            df_means[metric_name] += df[[col]].mean(axis=1)

    df_means = df_means.div(len(cols))

    # Rename columns for clarity
    rename_dict = {
        "factuality": "Human",
        "rouge_rouge_1_f_score": "ROUGE-1",
        "bertscore": "BERTScore",
        "rouge_rouge_2_f_score": "ROUGE-2",
        "rouge_rouge_l_f_score": "ROUGE-L",
        "bleu": "BLEU",
        "meteor": "METEOR",
        # "cosine_similarity": "Cosine Similarity",
        "ldfacts_src_hyp": "BARTScore",
        "bartscore_src_hyp": "LDFActs",
    }

    df_means = df_means.rename(columns=rename_dict)
    
    # Move 'LDFActs' to the last column
    if "LDFActs" in df_means.columns:
        cols = [col for col in df_means.columns if col != "LDFActs"] + ["LDFActs"]
        df_means = df_means[cols]
    
    # Calculate Kendall correlation matrix
    correlation_matrix = df_means.corr(method='kendall')

    # Set seaborn style and context
    sns.set(style="whitegrid", context="talk", font_scale=1.2)

    plt.figure(figsize=(12, 10))
    ax = sns.heatmap(
        correlation_matrix, 
        cmap="Blues",
        annot=True,
        fmt=".2f",
        xticklabels=correlation_matrix.columns,
        yticklabels=correlation_matrix.columns,
        annot_kws={"fontsize": 12},
        cbar_kws={"shrink": 0.8}
    )

    # Customize axis labels
    ax.set_xticklabels(ax.get_xticklabels(), rotation=90, horizontalalignment='right', fontsize=12)
    ax.set_yticklabels(ax.get_yticklabels(), fontsize=12)

    # Save figure
    plt.savefig(
        "./data/DialogSum/CoVe_metrics.png", 
        bbox_inches="tight", 
        dpi=300
    )
    plt.show()
    return correlation_matrix

# Call the function with your DataFrame
calculate_means_and_correlation(merged_df)

In [ ]:
# def kendal_tau_matrix(df):
#     df_new = get_df(df)
#     df_new = df_new.rename(
#         columns={
#             # "factuality": "Human",
#             "rouge_rouge_1_f_score": "ROUGE-1",
#             "bertscore": "BERTScore",
#             "rouge_rouge_2_f_score": "ROUGE-2",
#             "bartscore_src_hyp": "BARTScore",
#             "rouge_rouge_l_f_score": "ROUGE-L",
#             "bleu": "BLEU",
#             "meteor": "METEOR",
#             "fuzzywuzzy": "FuzzyWuzzy",
#             "cosine_similarity": "Cosine Similarity",
#             "ldfacts_src_hyp": "LongDocFACTScore",
#             # Add other metric renamings if necessary
#         }
#     )

#     cols_to_drop = ["coherence", "fluency"]
#     df_new = df_new.drop(columns=[col for col in cols_to_drop if col in df_new.columns])
#     cols = [
#         # "Human",
#         "LongDocFACTScore",
#         "ROUGE-1",
#         "ROUGE-2",
#         "ROUGE-L",
#         "BLEU",
#         "METEOR",
#         "FuzzyWuzzy",
#         "Cosine Similarity",
#         "BERTScore",
#         "BARTScore",
#         # Add "QuestEval", "FactCC" if you have them
#     ]
#     df_new = df_new[cols]
#     columns = df_new.columns
#     kendalltau_matrix = np.zeros(shape=(len(columns), len(columns)))
#     for idx1, col_n_1 in enumerate(df_new.columns):
#         for idx2, col_n_2 in enumerate(df_new.columns):
#             col_1 = np.array(df_new[col_n_1])
#             col_2 = np.array(df_new[col_n_2])
#             kendalltau_matrix[idx1, idx2] = np.round(
#                 scipy.stats.kendalltau(col_1, col_2)[0], 2
#             )
#     return df_new, kendalltau_matrix

In [ ]:
# # Calculate the Kendall tau matrix
# df_new, kt = kendal_tau_matrix(merged_df)

# plt.figure(figsize=(10, 8))
# sns.set(font_scale=1)
# plot = sns.heatmap(
#     kt,
#     cmap="Blues",
#     annot=True,
#     xticklabels=df_new.columns,
#     yticklabels=df_new.columns,
#     annot_kws={"fontsize": 8},
# )
# plt.savefig("aggreFACT_kendall.png", bbox_inches="tight")
# plt.show()

### LongDocFACTScore

In [ ]:
# merged_df = pd.merge(df_duc2004, df2, on='id', how='inner')
# merged_df.head()

In [ ]:
# ldfacts_columns = [col for col in df_duc2004.columns if 'ldfacts' in col]
# average_scores = df_duc2004[ldfacts_columns].mean()
# print(average_scores)

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

def visualize_correlation_matrix(df):
    cols = [
        "human_summary_factuality",
        "BART_summary_ldfacts_src_hyp",
        "T5_summary_ldfacts_src_hyp",
        "Pegasus_summary_ldfacts_src_hyp",
        "gpt4oMini_summary_ldfacts_src_hyp",
        "gpt35_summary_ldfacts_src_hyp",
        "gptInstruct_summary_ldfacts_src_hyp",
        "llama7bchat2_summary_ldfacts_src_hyp",
        "Orca7b2_summary_ldfacts_src_hyp",
        "Vic7b13_summary_ldfacts_src_hyp"
    ]

    rename_dict = {
        "human_summary_factuality": "Human",
        "BART_summary_ldfacts_src_hyp": "BART",
        "T5_summary_ldfacts_src_hyp": "T5",
        "Pegasus_summary_ldfacts_src_hyp": "Pegasus",
        "gpt4oMini_summary_ldfacts_src_hyp": "GPT-4o-Mini",
        "gpt35_summary_ldfacts_src_hyp": "GPT-3.5-Turbo",
        "gptInstruct_summary_ldfacts_src_hyp": "GPT-3.5-Turbo-Instruct",
        "llama7bchat2_summary_ldfacts_src_hyp": "LlaMA-2-7B-Chat",
        "Orca7b2_summary_ldfacts_src_hyp": "Orca-2-7B",
        "Vic7b13_summary_ldfacts_src_hyp": "Vicuna-v1.3-7B"
    }
    
    df = df[cols].rename(columns=rename_dict)
    cm = df.corr(method='kendall')
    
    plt.figure(figsize=(10, 8))
    sns.set(font_scale=1)
    sns.heatmap(
        cm,
        cmap="Blues",
        annot=True,
        xticklabels=df.columns,
        yticklabels=df.columns,
        annot_kws={"fontsize": 8},
    )
    
    # plt.xticks(rotation=90, ha='right')
    # plt.yticks(rotation=0)
    plt.savefig("aggreFACT_LLM_LDFacts.png", bbox_inches="tight") ### change here  
    plt.show()

visualize_correlation_matrix(merged_df)

#### Fleiss' Kappa

In [ ]:
def fleiss_kappa(M):
    N, k = M.shape
    n_annotators = float(np.sum(M[0, :]))
    tot_annotations = N * n_annotators
    category_sum = np.sum(M, axis=0)
    p = category_sum / tot_annotations
    PbarE = np.sum(p * p)
    P = (np.sum(M * M, axis=1) - n_annotators) / (n_annotators * (n_annotators - 1))
    Pbar = np.sum(P) / N
    return round((Pbar - PbarE) / (1 - PbarE), 4)

In [ ]:
def nominal_metric(a, b):
    return a != b

def interval_metric(a, b):
    return (a - b) ** 2

def ratio_metric(a, b):
    return ((a - b) / (a + b)) ** 2

def round_2dp(a):
    return np.round(a, 2)

#### Krippendorff Alpha

In [ ]:
def krippendorff_alpha(
    data,
    metric=interval_metric,
    force_vecmath=False,
    convert_items=float,
    missing_items=None,
):
    m = len(data)
    if missing_items is None:
        maskitems = []
    else:
        maskitems = list(missing_items)
    if np is not None:
        maskitems.append(np.ma.masked_singleton)
    units = {}
    for d in data:
        try:
            diter = d.items()
        except AttributeError:
            diter = enumerate(d)
        for it, g in diter:
            if g not in maskitems:
                try:
                    its = units[it]
                except KeyError:
                    its = []
                    units[it] = its
                its.append(convert_items(g))
    units = dict((it, d) for it, d in units.items() if len(d) > 1)
    n = sum(len(pv) for pv in units.values())
    if n == 0:
        raise ValueError("No items to compare.")
    np_metric = (np is not None) and ((metric in (interval_metric, nominal_metric, ratio_metric)) or force_vecmath)
    Do = 0.0
    for grades in units.values():
        if np_metric:
            gr = np.asarray(grades)
            Du = sum(np.sum(metric(gr, gri)) for gri in gr)
        else:
            Du = sum(metric(gi, gj) for gi in grades for gj in grades)
        Do += Du / float(len(grades) - 1)
    Do /= float(n)
    if Do == 0:
        return 1.0
    De = 0.0
    for g1 in units.values():
        if np_metric:
            d1 = np.asarray(g1)
            for g2 in units.values():
                De += sum(np.sum(metric(d1, gj)) for gj in g2)
        else:
            for g2 in units.values():
                De += sum(metric(gi, gj) for gi in g1 for gj in g2)
    De /= float(n * (n - 1))
    return round_2dp(1.0 - Do / De if (Do and De) else 1.0)